# 高光谱数据预处理主流程 / HSI Data Preprocessing Workflow

这份 Notebook 与 `scripts/运行数据预处理.py` 使用完全相同的七个接口，适合逐格理解和调试。

This notebook uses the same seven interfaces as `scripts/运行数据预处理.py` for step-by-step learning and debugging.

**安全边界 / Safety boundary**

- 不重新生成 train/validation/test 划分；
- 不重新拟合或覆盖标准化、PCA/LDA 状态；
- 不定义或训练分类模型；
- 不计算测试集性能指标。

In [ ]:
from pathlib import Path
import sys

import matplotlib
try:
    get_ipython().run_line_magic('matplotlib', 'inline')
except NameError:
    matplotlib.use('Agg')  # 无界面验收 / Headless verification
import matplotlib.pyplot as plt
plt.rcParams['font.sans-serif'] = ['Microsoft YaHei', 'SimHei', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False
import numpy as np
import torch

def find_project_root(start: Path) -> Path:
    for candidate in (start.resolve(), *start.resolve().parents):
        if (candidate / 'pyproject.toml').is_file():
            return candidate
    raise FileNotFoundError('Cannot locate hsi_project/pyproject.toml')

PROJECT_ROOT = find_project_root(Path.cwd())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from scripts.运行数据预处理 import (
    PAVIA_CLASS_NAMES_ZH,
    build_torch_dataloaders,
    build_torch_datasets,
    inspect_first_training_batch,
    load_fixed_dataset,
    load_frozen_preprocessing_state,
    load_workflow_settings,
    transform_full_cube,
    verify_frozen_split_relationships,
)
from src.visualization.预处理分析 import (
    plot_batch_class_distribution,
    plot_reducer_explained_variance,
    plot_split_spatial_map,
    plot_split_statistics,
    plot_training_mean_spectra,
    plot_training_patch_gallery,
    plot_training_reducer_scatter,
)

print(f'项目根目录 / Project root: {PROJECT_ROOT}')
print(f'PyTorch: {torch.__version__}')
print(f'CUDA 可用 / CUDA available: {torch.cuda.is_available()}')

## 步骤 1：加载配置 / Step 1: Load configuration

- **目的 / Purpose：** 固定数据协议、预处理路线与 DataLoader 参数。
- **输入 / Input：** 唯一日常入口 `Pavia数据预处理.yaml`。
- **输出 / Output：** `WorkflowSettings`。
- **验收 / Acceptance：** seed=345，路线名和冻结状态目录唯一。

In [ ]:
# 路径保持不变；只编辑统一 YAML 中的 split_protocol、reducer、representation。
# Keep this path fixed; edit the three selectors in the unified YAML.
CONFIG_PATH = PROJECT_ROOT / 'configs/数据预处理/Pavia数据预处理.yaml'
settings = load_workflow_settings(CONFIG_PATH, project_root=PROJECT_ROOT)
settings

## 步骤 2：读取原始数据与固定划分 / Step 2: Load raw data and frozen split

- **目的 / Purpose：** 建立光谱、二维坐标、标签与 split 身份的对应关系。
- **输入 / Input：** 只读 `.mat` 数据和已冻结 `.npz/.json` 划分。
- **输出 / Output：** `HSIDataBundle`。
- **验收 / Acceptance：** 42,776 个有标签像元被无重叠地完整覆盖。

In [ ]:
data = load_fixed_dataset(settings)
assert sum(len(v) for v in data.indices_by_split.values()) == len(data.labels)
total_pixels = data.label_map.size
labeled_pixels = len(data.labels)
background_pixels = total_pixels - labeled_pixels
print(f'总像元 / Total pixels: {total_pixels:,}')
print(f'有标签像元 / Labeled pixels: {labeled_pixels:,} ({labeled_pixels / total_pixels:.2%})')
print(f'背景或未标注 / Background or unlabeled: {background_pixels:,} ({background_pixels / total_pixels:.2%})')

### 检查 2C：固定划分约束与两协议关系 / Check 2C: Frozen split contract

该检查继承原固定划分 Notebook 中最重要的验收内容：当前协议三组无交集并覆盖全部有标签像元，每个非空 split 含九类；两协议样本身份相同、测试集相同，且 `fair.train ∪ fair.validation = paper30.train`。源文件哈希和同 seed 重建由 `scripts/验证固定划分.py` 与自动测试继续负责。

In [ ]:
split_checks = verify_frozen_split_relationships(settings, data)
assert all(value is True for key, value in split_checks.items() if key != 'current_protocol')

### 统计 2A：划分数量与空间位置 / Analysis 2A: Split counts and locations

这两幅统计只展示已经冻结的实验设计：总体/逐类样本数以及各 split 中心像元的位置。它们可以包含测试 split，因为没有查看测试光谱或预测结果。

In [ ]:
split_count_figure, _ = plot_split_statistics(data)
plt.show()

split_map_figure, _ = plot_split_spatial_map(data)
plt.show()

### 统计 2B：训练集逐类原始光谱 / Analysis 2B: Train-only class spectra

每个面板显示一个类别的训练光谱均值和 ±1 标准差。横轴是波段编号，不是实际波长；该图不读取验证或测试特征。

In [ ]:
spectral_figure, _ = plot_training_mean_spectra(
    data, class_names_zh=PAVIA_CLASS_NAMES_ZH
)
plt.show()

## 步骤 3：加载冻结预处理参数 / Step 3: Load frozen preprocessing state

- **目的 / Purpose：** 复用仅由训练中心像元拟合的标准化和 PCA/LDA 参数。
- **输入 / Input：** `preprocessing_state.npz` 与 `metadata.json`。
- **输出 / Output：** 已拟合的 `HSIPreprocessingPipeline`。
- **验收 / Acceptance：** 配置指纹和训练样本数匹配，验证/测试未参与拟合。

In [ ]:
pipeline = load_frozen_preprocessing_state(settings, data)
print(f'输出光谱维数 / Output bands: {pipeline.output_bands}')

## 步骤 4：变换完整立方体 / Step 4: Transform the full cube

- **目的 / Purpose：** 使用冻结参数把每个像元的 103 维光谱转换为配置指定的 PCA/LDA 分量。
- **输入 / Input：** `610×340×103` 原始立方体。
- **输出 / Output：** 内存中的 `610×340×B` 变换立方体（PCA15 时 B=15，LDA8 时 B=8）。
- **验收 / Acceptance：** 空间尺寸不变、float32、数值全部有限。

In [ ]:
transformed_cube = transform_full_cube(settings, data, pipeline)
assert transformed_cube.shape == (610, 340, settings.config.n_components)

In [ ]:
# 数据内容预览 / Data content preview (not a model result)
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
axes[0].imshow(data.cube[:, :, 50], cmap='gray')
axes[0].set_title('原始第51波段 / Raw band 51')
axes[1].imshow(transformed_cube[:, :, 0], cmap='viridis')
component_name = 'PCA' if settings.config.reducer == 'pca' else 'LDA'
axes[1].set_title(f'第一降维分量 / {component_name} component 1')
axes[2].imshow(data.label_map, cmap='tab10', vmin=0, vmax=9)
axes[2].set_title('标签图 / Ground-truth map')
for axis in axes:
    axis.axis('off')
plt.tight_layout()
plt.show()

### 统计 4A：冻结降维解释比 / Analysis 4A: Frozen reducer ratios

PCA 显示训练方差保留比例；LDA 显示各判别方向的相对判别解释比。二者都不是分类准确率。

In [ ]:
reducer_ratio_figure, _, _ = plot_reducer_explained_variance(pipeline)
plt.show()

### 统计 4B：训练集前两降维分量 / Analysis 4B: First two train-only components

每类最多固定抽取 500 个训练样本。PCA 观察最大方差方向，LDA 观察训练标签监督得到的判别方向；图中只使用训练样本。

In [ ]:
reducer_scatter_figure, _ = plot_training_reducer_scatter(
    data,
    pipeline,
    class_names_zh=PAVIA_CLASS_NAMES_ZH,
    max_samples_per_class=500,
    seed=345,
)
plt.show()

## 步骤 5：构造 Dataset / Step 5: Build datasets

- **目的 / Purpose：** 根据固定中心坐标按需提取 `25×25` 邻域。
- **输入 / Input：** 变换立方体、坐标、标签和 split 索引。
- **输出 / Output：** train/validation/test Dataset。
- **验收 / Acceptance：** 单样本输入为 `1×15×25×25`，身份字段互相对齐。

In [ ]:
datasets = build_torch_datasets(data, pipeline)
sample = datasets['train'][0]
print({key: tuple(value.shape) for key, value in sample.items()})

### 统计 5A：九类训练 patch 画廊 / Analysis 5A: Train patch gallery

每类选择一个靠近该类训练坐标中位位置的代表样本，显示第一主成分上的 25×25 邻域；红色加号是被分类的中心像元。

In [ ]:
patch_gallery_figure, _ = plot_training_patch_gallery(
    data,
    datasets,
    class_names_zh=PAVIA_CLASS_NAMES_ZH,
    component_index=0,
)
plt.show()

## 步骤 6：构造 DataLoader / Step 6: Build dataloaders

- **目的 / Purpose：** 以可复现顺序组织批数据。
- **输入 / Input：** Dataset 和 YAML 中的 batch/seed/worker 参数。
- **输出 / Output：** 三个 DataLoader。
- **验收 / Acceptance：** `drop_last=False`，所有样本都能遍历。

In [ ]:
loaders = build_torch_dataloaders(settings, data, pipeline)
print({name: None if loader is None else len(loader) for name, loader in loaders.items()})

## 步骤 7：检查模型输入批次 / Step 7: Inspect a model-input batch

- **目的 / Purpose：** 在模型开发前固定公开的数据接口。
- **输入 / Input：** train DataLoader 第一个 batch。
- **输出 / Output：** `input/label/raw_label/coordinate/sample_index`。
- **验收 / Acceptance：** 输入为 `N×1×15×25×25`；模型标签为 0–8，原标签为 1–9。

In [ ]:
batch = inspect_first_training_batch(settings, loaders)
print('模型以后只需要读取 batch[\"input\"] 和 batch[\"label\"]')
print('Future models only need batch[\"input\"] and batch[\"label\"]')

In [ ]:
# 首个训练 batch 的类别构成 / Class composition of the first train batch
batch_distribution_figure, _ = plot_batch_class_distribution(
    batch,
    class_names_en=data.spec.class_names,
    class_names_zh=PAVIA_CLASS_NAMES_ZH,
)
plt.show()

## 如何阅读这些统计图 / How to interpret the visual statistics

1. **类别不均衡 / Class imbalance：** Meadows 的样本明显多于 Shadows、Painted metal sheets 等类别。后续不能只看 OA，还应报告 AA、逐类准确率和混淆矩阵。是否使用类别权重必须在验证集上决定。
2. **光谱差异 / Spectral differences：** 观察九类训练均值曲线在哪些波段分离，以及标准差范围是否大量重叠。这只能形成波段选择假设，不能直接证明分类效果。
3. **降维解释比 / Reducer ratios：** PCA 方差比与 LDA 判别解释比含义不同；二者都不等于准确率。
4. **降维散点 / Reducer scatter：** PCA 不使用标签，LDA 只用训练标签；二维投影不能替代正式分类评价。
5. **空间邻域 / Spatial patches：** 比较不同类别的形状、纹理和边界。红色中心是当前标签归属像元，邻域内其他像元的标签没有被用于输入。
6. **batch 构成 / Batch composition：** 首个 batch 是训练集随机打乱后的一个样本，不会严格保持 24%/6%/70% 或逐类均衡，也不能代表整个训练集。
7. **安全边界 / Safety boundary：** split 数量和位置可展示全部划分；特征内容分析只使用训练集。测试集特征、预测和指标应留到最终配置冻结以后。

---

## 本阶段输出 / Output of this stage

现在已经得到后续模型可以共享的公开接口：

```python
inputs = batch['input']   # N×1×B×25×25, float32；PCA15: B=15，LDA8: B=8
labels = batch['label']   # N, int64, values 0..8
```

`raw_label`、`coordinate` 和 `sample_index` 用于结果追溯和分类图回填。下一阶段的模型定义应放在 `src/models/`，不得反向修改这里的划分或标签。